# **Install dependencies**

In [0]:
%pip install --quiet "psycopg[binary]" pymongo pandas

## **Inventory PostgreSQL and MongoDB**

In [0]:
# ============================================================
# BANGLAPAY | PHASE 1
# POSTGRESQL + MONGODB SOURCE INVENTORY
# Environment: DEV
# Read-only metadata inspection
# ============================================================

import pandas as pd
import psycopg

from pymongo import MongoClient
from psycopg.rows import dict_row


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SCOPE = "banglapay-ingestion"

POSTGRES_DATABASE = "banglapay"
MONGODB_DATABASE = "banglapay_ops"

ENVIRONMENT = "dev"

# Retrieve credentials without displaying them.
PG_JDBC_URL = dbutils.secrets.get(
    SCOPE, "NEON_JDBC_URL"
)

PG_USERNAME = dbutils.secrets.get(
    SCOPE, "NEON_USERNAME"
)

PG_PASSWORD = dbutils.secrets.get(
    SCOPE, "NEON_PASSWORD"
)

MONGO_URI = dbutils.secrets.get(
    SCOPE, "MONGODB_CONNECTION_URI"
)


# ------------------------------------------------------------
# 2. PostgreSQL inventory
# ------------------------------------------------------------

pg_url = (
    PG_JDBC_URL[len("jdbc:"):]
    if PG_JDBC_URL.startswith("jdbc:")
    else PG_JDBC_URL
)

pg_query = """
SELECT
    t.table_schema,
    t.table_name,

    COALESCE(
        (
            SELECT array_agg(
                kcu.column_name
                ORDER BY kcu.ordinal_position
            )
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
             AND tc.table_name = kcu.table_name
            WHERE tc.constraint_type = 'PRIMARY KEY'
              AND tc.table_schema = t.table_schema
              AND tc.table_name = t.table_name
        ),
        ARRAY[]::text[]
    ) AS primary_key_columns,

    COALESCE(
        (
            SELECT array_agg(
                c.column_name
                ORDER BY c.ordinal_position
            )
            FROM information_schema.columns c
            WHERE c.table_schema = t.table_schema
              AND c.table_name = t.table_name
              AND c.data_type IN (
                  'timestamp without time zone',
                  'timestamp with time zone',
                  'date'
              )
              AND (
                  c.column_name ILIKE '%created%'
                  OR c.column_name ILIKE '%updated%'
                  OR c.column_name ILIKE '%modified%'
                  OR c.column_name ILIKE '%timestamp%'
              )
        ),
        ARRAY[]::text[]
    ) AS timestamp_candidates

FROM information_schema.tables t

WHERE t.table_type = 'BASE TABLE'
  AND t.table_schema NOT IN (
      'pg_catalog',
      'information_schema'
  )

ORDER BY
    t.table_schema,
    t.table_name;
"""

with psycopg.connect(
    pg_url,
    user=PG_USERNAME,
    password=PG_PASSWORD,
    connect_timeout=15,
    sslmode="require",
    row_factory=dict_row
) as connection:

    with connection.cursor() as cursor:

        # Strictly read-only metadata query.
        cursor.execute("SET TRANSACTION READ ONLY")

        cursor.execute(pg_query)

        pg_inventory = [
            dict(row)
            for row in cursor.fetchall()
        ]


pg_df = pd.DataFrame(pg_inventory)

print("========== NEON POSTGRESQL ==========")
print("Discovered tables:", len(pg_inventory))

display(pg_df)


# ------------------------------------------------------------
# 3. MongoDB collection inventory
# ------------------------------------------------------------

mongo_client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=15000,
    connectTimeoutMS=10000,
    socketTimeoutMS=15000
)

try:

    mongo_client.admin.command("ping")

    mongo_db = mongo_client[MONGODB_DATABASE]

    collection_names = sorted(
        mongo_db.list_collection_names()
    )

    mongo_inventory = [
        {
            "database": MONGODB_DATABASE,
            "collection": name,
            "primary_key": "_id"
        }
        for name in collection_names
    ]

finally:

    mongo_client.close()


mongo_df = pd.DataFrame(mongo_inventory)

print("========== MONGODB ATLAS ==========")
print("Discovered collections:", len(mongo_inventory))

display(mongo_df)


# ------------------------------------------------------------
# 4. Inventory validation
# ------------------------------------------------------------

print("========== INVENTORY SUMMARY ==========")

print("Environment:", ENVIRONMENT.upper())
print("PostgreSQL tables:", len(pg_inventory))
print("MongoDB collections:", len(mongo_inventory))

print(
    "PostgreSQL missing primary keys:",
    sum(
        not row["primary_key_columns"]
        for row in pg_inventory
    )
)

if len(pg_inventory) != 22:
    print("WARNING: PostgreSQL expected 22 tables.")

if len(mongo_inventory) != 6:
    print("WARNING: MongoDB expected 6 collections.")

print("SOURCE INVENTORY COMPLETED")

## **Missing timestamp column discovery**

In [0]:
# ============================================================
# Inspect tables without detected timestamp candidates
# Read-only metadata inspection
# ============================================================

import pandas as pd
import psycopg
from psycopg.rows import dict_row

query = """
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    udt_name,
    is_nullable,
    column_default

FROM information_schema.columns

WHERE
    (table_schema = 'integration'
     AND table_name = 'source_runs')

    OR

    (table_schema = 'payment'
     AND table_name = 'transaction_status_history')

ORDER BY
    table_schema,
    table_name,
    ordinal_position
"""

with psycopg.connect(
    pg_url,
    user=PG_USERNAME,
    password=PG_PASSWORD,
    sslmode="require",
    row_factory=dict_row
) as conn:

    with conn.cursor() as cur:

        cur.execute("SET TRANSACTION READ ONLY")
        cur.execute(query)

        columns = [
            dict(row)
            for row in cur.fetchall()
        ]

display(pd.DataFrame(columns))

## **Complete extraction metadata**

In [0]:
# ============================================================
# BANGLAPAY | PHASE 1
# COMPLETE BATCH SOURCE METADATA
# Read-only inspection
# ============================================================

import pandas as pd
import psycopg
from psycopg.rows import dict_row
from pymongo import MongoClient


SCOPE = "banglapay-ingestion"

pg_jdbc_url = dbutils.secrets.get(SCOPE, "NEON_JDBC_URL")
pg_username = dbutils.secrets.get(SCOPE, "NEON_USERNAME")
pg_password = dbutils.secrets.get(SCOPE, "NEON_PASSWORD")

mongo_uri = dbutils.secrets.get(
    SCOPE, "MONGODB_CONNECTION_URI"
)

mongo_database = "banglapay_ops"

pg_url = (
    pg_jdbc_url[len("jdbc:"):]
    if pg_jdbc_url.startswith("jdbc:")
    else pg_jdbc_url
)


# ------------------------------------------------------------
# 1. PostgreSQL temporal columns across all source tables
# ------------------------------------------------------------

temporal_query = """
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable

FROM information_schema.columns

WHERE table_schema NOT IN (
    'pg_catalog',
    'information_schema'
)

AND data_type IN (
    'timestamp with time zone',
    'timestamp without time zone',
    'date'
)

ORDER BY
    table_schema,
    table_name,
    ordinal_position
"""


# ------------------------------------------------------------
# 2. Complete Outbox metadata
# ------------------------------------------------------------

outbox_query = """
SELECT
    ordinal_position,
    column_name,
    data_type,
    udt_name,
    is_nullable,
    column_default

FROM information_schema.columns

WHERE table_schema = 'integration'
  AND table_name = 'outbox_events'

ORDER BY ordinal_position
"""


with psycopg.connect(
    pg_url,
    user=pg_username,
    password=pg_password,
    sslmode="require",
    connect_timeout=15,
    row_factory=dict_row
) as connection:

    with connection.cursor() as cursor:

        cursor.execute("SET TRANSACTION READ ONLY")

        cursor.execute(temporal_query)
        temporal_columns = [
            dict(row) for row in cursor.fetchall()
        ]

        cursor.execute(outbox_query)
        outbox_columns = [
            dict(row) for row in cursor.fetchall()
        ]


print("========== POSTGRESQL TEMPORAL COLUMNS ==========")
display(pd.DataFrame(temporal_columns))

print("========== OUTBOX TABLE STRUCTURE ==========")
display(pd.DataFrame(outbox_columns))


# ------------------------------------------------------------
# 3. MongoDB collection field/type inventory
# Inspect field names and BSON types only.
# Never display document values.
# ------------------------------------------------------------

mongo_client = MongoClient(
    mongo_uri,
    serverSelectionTimeoutMS=15000,
    connectTimeoutMS=10000
)

mongo_field_inventory = []

try:

    mongo_client.admin.command("ping")

    mongo_db = mongo_client[mongo_database]

    collection_names = sorted(
        mongo_db.list_collection_names()
    )

    for collection_name in collection_names:

        pipeline = [
            {"$limit": 200},

            {
                "$project": {
                    "fields": {
                        "$objectToArray": "$$ROOT"
                    }
                }
            },

            {"$unwind": "$fields"},

            {
                "$group": {
                    "_id": {
                        "field": "$fields.k",
                        "bson_type": {
                            "$type": "$fields.v"
                        }
                    },
                    "sampled_documents": {
                        "$sum": 1
                    }
                }
            },

            {
                "$sort": {
                    "_id.field": 1,
                    "_id.bson_type": 1
                }
            }
        ]

        results = list(
            mongo_db[collection_name].aggregate(
                pipeline,
                allowDiskUse=False
            )
        )

        for item in results:

            mongo_field_inventory.append({
                "collection": collection_name,
                "field": item["_id"]["field"],
                "bson_type": item["_id"]["bson_type"],
                "sampled_documents": item[
                    "sampled_documents"
                ]
            })

finally:

    mongo_client.close()


print("========== MONGODB FIELD INVENTORY ==========")
display(pd.DataFrame(mongo_field_inventory))

print("========== INVENTORY COMPLETE ==========")
print("MongoDB collections:", len(collection_names))
print("MongoDB fields discovered:", len(mongo_field_inventory))